# Colab Demo: Linear Optics Toolkit

这个 notebook 按 state-first 的 5 步流程演示：
1. 态构建
2. 态运算（经典混合 / 相干叠加）
3. 态演化
4. 不变量计算
5. 态测量


In [ ]:
import importlib.util
import subprocess
import sys

REPO_PIP = 'git+https://github.com/yangbc30/hierarchical-invariants-paper-code.git'
if importlib.util.find_spec('photonic_jordan') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', REPO_PIP])
print('photonic_jordan import availability:', importlib.util.find_spec('photonic_jordan') is not None)


## 1) 态构建


In [ ]:
from photonic_jordan import Fock, FockMixed, from_modes_and_gram

rho = from_modes_and_gram([0, 1, 0], gram=0.4, m_ext=2)
rho_fock = Fock(2, 1, 0)
rho_mix = FockMixed((0.6, 2, 1, 0), (0.4, 1, 2, 0))

print('rho:', rho)
print('rho_fock:', rho_fock)
print('rho_mix:', rho_mix)


## 2) 态运算（经典混合 / 相干叠加）


In [ ]:
import numpy as np
from photonic_jordan import Fock, from_modes_and_gram

rho_a = from_modes_and_gram([0, 1, 0], gram=0.4, m_ext=2)
rho_b = from_modes_and_gram([0, 1, 0], gram=0.2, system=rho_a.system)

# classical mixture
rho_classical = rho_a.mix(rho_b, weight=0.3)

# coherent superposition (pure states only)
rho_p = Fock(1, 0)
rho_q = Fock(0, 1, system=rho_p.system)
rho_coherent = rho_p.superpose(rho_q, alpha=np.sqrt(0.8), beta=np.sqrt(0.2) * np.exp(1j * np.pi / 3))

print('classical trace:', np.trace(rho_classical.density_matrix(rep='tensor')))
print('coherent trace:', np.trace(rho_coherent.density_matrix(rep='fock')))


## 3) 态演化


In [ ]:
S = rho.system.unitary.haar(seed=123)
rho_out = rho.evolve(S)
print('evolved:', rho_out)


## 4) 不变量计算


In [ ]:
print('I_exact(1):', rho_out.invariant.I_exact(1))
print('I_cumulative(2):', rho_out.invariant.I_cumulative(2))
print('I_exact(1, sector=(2,1)):', rho_out.invariant.I_exact(1, sector=(2, 1)))
print('I_exact(1, multiplicity=((2,1),0)):', rho_out.invariant.I_exact(1, multiplicity=((2, 1), 0)))


## 5) 态测量


In [ ]:
obs = rho_out.system.observable.sigma_z(modes=(0, 1))

print('expectation:', rho_out.measure.expectation(obs))
print('variance:', rho_out.measure.variance(obs))
print('distribution:', rho_out.measure.distribution(obs))
